In [1]:
# Transformers example -> Sorting a sequence of numbers
%reload_ext autoreload
%autoreload 2

In [2]:
import flax.nnx as nnx
import jax
import jax.numpy as jnp
import jax.random as jrandom
import optax

from probjax.nn.nets.lru import LRUModel
from probjax.nn.nets.transformer import PosEmbed

In [3]:
VOCAB_SIZE = 10

In [4]:
def generate_data(key, n, T, vocab_size=10):
    sequences = jrandom.randint(key, (n, T,1), 0, vocab_size, dtype=jnp.int32)

    sequences_sorted = jnp.sort(sequences, axis=-2)

    return sequences, sequences_sorted

inputs, labels = generate_data(jax.random.PRNGKey(0), 1000, 10, VOCAB_SIZE)

2024-10-29 14:46:00.572109: W external/xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.5 which is older than the PTX compiler version 12.6.77. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [5]:
key = jrandom.PRNGKey(0)

In [6]:
class Model(nnx.Module):

    def __init__(self, dim,rngs, dropout_rate=0.1):
        self.embed = nnx.Embed(VOCAB_SIZE, dim, rngs=rngs)
        self.pos_embed = PosEmbed(dim, 100,rngs=rngs)
        self.transformer = LRUModel(dim,dim, dim,4, rngs)
        self.output = nnx.Linear(dim, VOCAB_SIZE, rngs=rngs)

    def __call__(self, x, deterministic=False):
        x = self.embed(x)
        x = jnp.squeeze(x,axis=-2)
        x = self.pos_embed(x)
        x = self.transformer(x,deterministic)
        x = self.output(x)
        return x



In [7]:
model = Model(100, rngs=nnx.Rngs(0), dropout_rate=None)

In [8]:
nnx.display(model)

In [9]:
params = nnx.state(model, nnx.Param)

In [10]:
optimizer = optax.adam(5e-4)
opt_state = optimizer.init(params)

In [11]:

train_seq_len = [5,10,50]
def loss_fn(params,model, key):
    nnx.update(model, params)
    l = 0
    for t in train_seq_len:
        key, key_sub = jax.random.split(key)
        inp_data, labels = generate_data(key_sub, 128, t, vocab_size=VOCAB_SIZE)
        logits = model(inp_data)
        labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
        loss = optax.softmax_cross_entropy(logits, labels).mean()
        l += loss
    return l

@jax.jit
def acc(params,model, inputs, outputs):
    nnx.update(model, params)
    inp_data, labels = inputs, outputs
    logits = model(inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    acc = (logits.argmax(axis=-1) == labels.argmax(-1)).mean()
    return acc

@jax.jit
def update(params,model, key, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params,model, key)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state

In [12]:
model.train()

In [13]:
key = jrandom.PRNGKey(0)

In [14]:
test_seq = 10
for i in range(10_000):
    key, subkey, key2 = jrandom.split(key, 3)
    loss, params, opt_state = update(params,model,key, opt_state)
    if (i % 1000) == 0:
        inputs, labels = generate_data(key, 32,test_seq, vocab_size=VOCAB_SIZE)
        accuracy = acc(params,model, inputs, labels)
        print(accuracy, loss)

/root/miniconda3/envs/probjax_new/lib/python3.12/site-packages/jax/_src/lax/lax.py:3227: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)
/root/miniconda3/envs/probjax_new/lib/python3.12/site-packages/jax/_src/lax/lax.py:3227: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)
/root/miniconda3/envs/probjax_new/lib/python3.12/site-packages/jax/_src/lax/lax.py:3227: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


0.215625 7.28607


KeyboardInterrupt: 

In [106]:
model.eval()
nnx.update(model, params)

In [118]:
input = jax.random.randint(key+1, (1, 50,1),0, 10,dtype=jnp.int32)
outputs = model(input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])
print(jnp.sort(input[0,...,0]))

[7 6 5 7 7 4 6 0 0 2 6 5 1 8 2 7 8 9 8 0 9 7 6 7 4 3 1 8 4 4 1 5 8 9 9 3 5
 1 6 7 9 0 8 8 7 0 2 1 1 7]
[0 0 0 0 0 1 1 1 1 1 1 2 2 2 3 3 4 4 4 4 5 5 5 5 6 6 6 6 6 7 7 7 7 7 7 7 7
 8 8 8 8 8 8 8 8 9 9 9 9 9]
[0 0 0 0 0 1 1 1 1 1 1 2 2 2 3 3 4 4 4 4 5 5 5 5 6 6 6 6 6 7 7 7 7 7 7 7 7
 7 8 8 8 8 8 8 8 9 9 9 9 9]


In [119]:
jnp.allclose(outputs.argmax(-1)[0], jnp.sort(input[0,...,0]))

Array(False, dtype=bool)

In [ ]:
input = jax.random.randint(key, (1, 10,1),0, 10,dtype=jnp.int32)
outputs = f.apply(params, key + 2, input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])

[0 0 2 1 9 1 3 0 6 5]
[0 0 0 1 1 2 3 5 6 9]
